# Section 1. Install deps


In [ ]:
!pip install transformers torch accelerate

# Section 2. Data / test cases


In [1]:
test_cases = [
  {
    "case_id": "case_001",
    "case_type": "simple_case",
    "top_words": ["НАТО", "альянс", "оборона", "саміт", "Брюссель"],
    "top_docs": ["Лідери НАТО зібралися на саміт у Брюсселі", "Питання колективної оборони обговорено на зустрічі альянсу"],
    "expected": {"classification": "politics", "quality": "good", "is_noise": False, "noise_words": []}
  },
  {
    "case_id": "case_002",
    "case_type": "missing_data",
    "top_words": [],
    "top_docs": ["Ціни на нафту зросли на 5%", "Барель марки Brent торгується за новою ціною"],
    "expected": {"classification": "economy", "quality": "mixed/bad", "is_noise": False, "noise_words": []}
  },
  {
    "case_id": "case_003",
    "case_type": "noisy_text",
    "top_words": ["та", "що", "сказав", "це", "року", "воно", "їх", "було"],
    "top_docs": ["Компанія X-Corp (колишня Twitter) уклала угоду з приватною фірмою в ОАЕ", "В Україні торік зареєстрували 267 видавців і книгарень"],
    "expected": {"classification": "economy/culture", "quality": "bad", "is_noise": True, "noise_words": ["та", "що", "сказав", "це", "року", "воно", "їх", "було"]}
  },
  {
    "case_id": "case_004",
    "case_type": "empty_tool_result",
    "top_words": ["qwe12", "xyz_4", "10101", "err_log", "як"],
    "top_docs": ["Системне повідомлення qwe12", "Лог помилок 10101"],
    "expected": {"classification": "-", "quality": "bad", "is_noise": True, "noise_words": ["як"]}
  },
  {
    "case_id": "case_005",
    "case_type": "no_redundant_tools",
    "top_words": ["футбол", "чемпіонат", "гол", "стадіон", "ФІФА"],
    "top_docs": ["Фінал чемпіонату світу", "Забитий гол на стадіоні"],
    "expected": {"classification": "sports", "quality": "good", "is_noise": False, "noise_words": []}
  },
  {
    "case_id": "case_006",
    "case_type": "ambiguity",
    "top_words": ["ядро", "процесор", "клітина", "біологія", "Intel", "синтез"],
    "top_docs": ["Новий процесор Intel має 16 ядер", "Ядро клітини містить генетичний код"],
    "expected": {"classification": "technology/biology/science", "quality": "mixed", "is_noise": False, "noise_words": []}
  },
  {
    "case_id": "case_007",
    "case_type": "sequential_tools",
    "top_words": ["гривня", "НБУ", "інфляція", "курс", "долар"],
    "top_docs": ["НБУ встановив офіційний курс", "Інфляція в Україні сповільнилася"],
    "expected": {"classification": "economy", "quality": "good", "is_noise": False, "noise_words": []}
  },
  {
    "case_id": "case_008",
    "case_type": "validator_issue",
    "top_words": ["вакцина", "щеплення", "вірус", "пандемія", "МОЗ"],
    "top_docs": ["Нова партія вакцини прибула", "Рецепт смачного борщу з пампушками"],
    "expected": {"classification": "health", "quality": "mixed", "is_noise": False, "noise_words": []}
  },
  {
    "case_id": "case_009",
    "case_type": "reference_to_output",
    "top_words": ["Марс", "NASA", "висадка", "космос", "запуск"],
    "top_docs": ["NASA запускає найбільше дослідження центру Чумацького Шляху", "Перший крок до висадки людини на Місяць"],
    "expected": {"classification": "science", "quality": "good", "is_noise": False, "noise_words": []}
  },
  {
    "case_id": "case_010",
    "case_type": "agent_error_correction",
    "top_words": ["Apple", "смартфон", "iPhone", "iOS", "садівництво"],
    "top_docs": ["Як виростити яблука в саду", "Обрізка дерев восени"],
    "expected": {"classification": "technology/gardening", "quality": "mixed", "is_noise": False, "noise_words": []}
  }
]

# Section 3. Tool call logger


In [2]:
import json
import torch
from datetime import datetime
from functools import wraps
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import re
import ast

tool_usage_history = []

def log_tool_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        tool_name = func.__name__
        param_names = func.__code__.co_varnames[:len(args)]
        input_data = {**dict(zip(param_names, args)), **kwargs}
        log_entry = {
            "timestamp": datetime.now().isoformat(timespec='seconds'),
            "tool_name": tool_name,
            "input": input_data,
            "output": None,
            "success": False,
            "error": None,
            "reason": ""
        }

        try:
            result = func(*args, **kwargs)
            log_entry.update({"output": result, "success": True})

            if tool_name == "detect_noise_words":
                log_entry["reason"] = f"Found {result.get('noise_count', 0)} noise words."
            elif tool_name == "score_topic_quality":
                log_entry["reason"] = f"Quality score is {result.get('quality_score')}."
            else:
                log_entry["reason"] = "Task completed."

            return result
        except Exception as e:
            log_entry["error"] = str(e)
            log_entry["reason"] = "Technical failure."
            return {"error": str(e)}
        finally:
            tool_usage_history.append(log_entry)

    return wrapper

# Section 4. Tool definitions


In [3]:
domains = {

    "technology": {
        "intel", "amd", "nvidia", "процесор", "cpu", "gpu",
        "ядро", "сервер", "память", "компьютер",
        "ноутбук", "мережа", "алгоритм", "ai",
        "штучний", "дані", "програмування", "код",
        "python", "java", "linux", "windows",
        "база", "database", "cloud", "кібербезпека",
        "чип", "робот", "автоматизація", "інтернет",
        "веб", "software", "hardware"
    },

    "biology": {
        "клітина", "генетичний", "біологія",
        "днк", "рнк", "ядро", "синтез",
        "організм", "бактерія", "вірус",
        "еволюція", "ген", "мікроб",
        "тканина", "фермент", "мутація",
        "молекула", "екосистема", "фотосинтез",
        "біохімія", "нейрон", "хромосома"
    },

    "science": {
        "дослідження", "синтез",
        "аналіз", "експеримент",
        "теорія", "модель", "фізика",
        "хімія", "математика", "формула",
        "статистика", "лабораторія",
        "спостереження", "гіпотеза",
        "відкриття", "квантовий",
        "енергія", "реакція"
    },

    "economy": {
        "економіка", "ринок", "банк",
        "інфляція", "валюта", "гривня",
        "долар", "євро", "кредит",
        "податок", "інвестиція", "бюджет",
        "фінанси", "акції", "біржа", "нафту",
        "бізнес", "компанія", "прибуток",
        "зарплата", "експорт", "імпорт",
        "торгівля", "капітал", "ціна", "ціни"
    },

    "politics": {
        "уряд", "парламент", "президент",
        "вибори", "закон", "політика",
        "депутат", "міністр", "санкції",
        "нато", "альянс", "влада",
        "конституція", "реформа",
        "дипломатія", "саміт",
        "війна", "оборона", "держава",
        "голосування", "партія"
    },

    "culture": {
        "культура", "мистецтво",
        "музика", "театр", "кіно",
        "фільм", "література", "роман",
        "поезія", "традиція", "музей",
        "виставка", "художник",
        "актор", "концерт", "фестиваль",
        "танець", "пісня", "архітектура",
        "спадщина"
    },

    "sports": {
        "спорт", "футбол", "баскетбол",
        "теніс", "матч", "турнір",
        "олімпіада", "команда",
        "чемпіонат", "гол", "тренер",
        "гравець", "стадіон", "медаль",
        "перемога", "поразка",
        "біг", "фітнес", "рекорд",
        "ліга"
    },

    "health": {
        "здоров’я", "лікар", "хвороба",
        "пацієнт", "медицина", "лікування",
        "вакцина", "вірус", "епідемія",
        "симптом", "діагноз",
        "терапія", "операція",
        "імунітет", "психологія",
        "харчування", "стрес",
        "реабілітація", "клініка",
        "антибіотик"
    },

    "gardening": {
        "сад", "город", "рослина", "квітка",
        "дерево", "кущ", "насіння", "ґрунт",
        "полив", "добриво", "врожай",
        "теплиця", "садівництво", "газон",
        "троянда", "тюльпан", "томат",
        "огірок", "картопля", "бур’ян",
        "лопата", "граблі", "компост",
        "мульча", "саджанці", "пересадка",
        "обрізка", "земля", "овочі",
        "фрукти", "ягоди", "перець",
        "морква", "цибуля", "зелень"
    }
}


@log_tool_call
def detect_noise_words(top_words: list) -> dict:
    if not top_words:
        raise ValueError("Empty top_words and top_docs lists provided")
    stop_words = {"та", "що", "це", "був", "було", "воно", "їх", "його", "але", "для", "про", "як", "сказав", "року"}
    noise = [word for word in top_words if word.lower() in stop_words or len(word) < 2]
    is_noise = False
    if noise: is_noise = True
    return {"is_noise": is_noise, "noise_words": noise}

@log_tool_call
def suggest_topic_label(top_words: list, top_docs: list) -> dict:
    if not top_words and not top_docs:
        raise ValueError("Empty top_words and top_docs lists provided")

    words_lower = [w.lower() for w in top_words]
    docs_text = " ".join(top_docs).lower()

    domain_scores = {}
    for domain, keywords in domains.items():
        score = 0
        for word in words_lower:
            if word in keywords:
                score += 1
        for kw in keywords:
            if kw in docs_text:
                score += 1
        domain_scores[domain] = score

    active_domains = [
        domain
        for domain, score in domain_scores.items()
        if score > 0
    ]

    if not active_domains:
        classification = "unknown"
    else:
        classification = "/".join(active_domains)

    return {"classification": classification}


@log_tool_call
def score_topic_quality(top_words: list, top_docs: list) -> dict:
    if not top_words or not top_docs:
        return {"quality": "bad", "score": 0.0}

    words_lower = [w.lower() for w in top_words]
    docs_text = " ".join(top_docs).lower()

    domain_scores = {}

    for domain, keywords in domains.items():
        score = 0
        for word in words_lower:
            if word in keywords:
                score += 1
        for kw in keywords:
            if kw in docs_text:
                score += 1
        domain_scores[domain] = score

    active_domains = [
        domain
        for domain, score in domain_scores.items()
        if score > 0
    ]

    if len(active_domains) == 1:
        quality = "good"
    elif len(active_domains) > 1:
        quality = "mixed"
    else:
        quality = "bad"

    return {"quality": quality}


tools_map = {
    "detect_noise_words": detect_noise_words,
    "suggest_topic_label": suggest_topic_label,
    "score_topic_quality": score_topic_quality
}

# Section 5. Agent design


In [4]:
import json
import torch
import ast
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

class TopicAgent:
    def __init__(self, model_name="Qwen/Qwen2.5-1.5B-Instruct"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="cpu"
        )
        self.pipe = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer
        )

        self.system_prompt_baseline = """You are a Topic Modeling Assistant.
        You MUST return ONLY valid JSON with the following fields:
        - classification (topic titles (one, several, "unknown"))
        - quality (one topic -> "good", two or more topic -> "mixed", "unknown" -> "bad")
        - is_noise (one or more noise words -> True, zero noise words -> False),
        - noise_words (list of noise words)

        Output schema:
        {
          "classification": lowercase string,
          "quality": lowercase string,
          "is_noise": boolean,
          "noise_words": list or null
        }
        """

        self.system_prompt_tools = """You are a Topic Modeling Assistant.
        RUN all these 3 tools:
        - detect_noise_words(top_words: list): Returns noise/stop words.
        - suggest_topic_label(top_words: list, top_docs: list): Returns a label.
        - score_topic_quality(top_words: list, top_docs: list): Returns a quality score (0 to 1).

        Format:
        Thought: your reasoning.
        Action: tool_name({"param": "value"})
        Observation: tool output.
        ... (repeat if needed)
        Final Answer: your conclusion.
        """

# Section 6. Baseline LLM without tools


In [5]:
def safe_parse(text):
    text = re.sub(r"```json\s*", "", text)
    text = re.sub(r"```", "", text).strip()

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON found in: {text}")

    return json.loads(match.group(0))

def run_baseline(self, user_input):
        messages = [
            {"role": "system", "content": self.system_prompt_baseline},
            {"role": "user", "content": user_input}
        ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        outputs = self.pipe(
            text,
            max_new_tokens=512,
            do_sample=False,
            return_full_text=False
        )

        response = outputs[0]["generated_text"].strip()

        try:
            return safe_parse(response)
        except json.JSONDecodeError:
            return {"error": "Failed to parse JSON", "raw_response": response}

TopicAgent.run_baseline = run_baseline

# Section 7. Agent with tools


In [7]:
def run_with_tools(self, user_input):
        messages = [{"role": "system", "content": self.system_prompt_tools}, {"role": "user", "content": user_input}]

        for _ in range(1):
            prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            output = self.pipe(prompt, max_new_tokens=256, do_sample=False)[0]['generated_text']
            response = output.split("<|im_start|>assistant\n")[-1]
            all_observations = []
            if "Action:" in response:
                try:
                    action_lines = [
                        line.strip()
                        for line in response.splitlines()
                        if line.strip().startswith("Action:")
                    ]

                    current_step_observations = []

                    for line in action_lines:
                        action_content = line[len("Action:"):].strip()
                        tool_name = action_content.split("(", 1)[0].strip()
                        args_str = action_content.split("(", 1)[1].rsplit(")", 1)[0]
                        parsed_args = ast.literal_eval(args_str)

                        if tool_name == "detect_noise_words":
                            observation = tools_map[tool_name](parsed_args)
                        else:
                            observation = tools_map[tool_name](*parsed_args)

                        obs_entry = {"tool": tool_name, "observation": observation}
                        all_observations.append(obs_entry)
                        current_step_observations.append(obs_entry)

                    messages.append({"role": "assistant", "content": response})
                    messages.append({
                        "role": "user",
                        "content": f"Observations: {json.dumps(current_step_observations, ensure_ascii=False)}"
                    })

                except Exception as e:
                    messages.append({"role": "user", "content": f"Observation: Error: {e}"})
            if "Final Answer:" in response:
                return response.split("Final Answer:")[-1].strip(), all_observations
        return "Limit reached", all_observations

TopicAgent.run_with_tools = run_with_tools

# Section 8. Run 10 test cases


In [8]:
agent = TopicAgent()

baseline_results = []
tools_results = []
final_answers = []

for item in test_cases:
  input_data = f"""
  Analyze this topic:
  Words: {item.get("top_words")},
  Docs: {item.get("top_docs")}
  """

  final_res = agent.run_baseline(input_data)
  baseline_results.append(final_res)
  print(f"\n=== РЕЗУЛЬТАТ {item["case_id"]} (baseline) ===\n{final_res}")

  final_res, obs = agent.run_with_tools(input_data)
  tools_results.append(obs)
  final_answers.append(final_res)
  print(f"\n=== РЕЗУЛЬТАТ {item["case_id"]} (with tools) ===\n{obs}")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...


=== РЕЗУЛЬТАТ case_009 (baseline) ===
{'classification': 'space exploration', 'quality': 'good', 'is_noise': False, 'noise_words': []}


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== РЕЗУЛЬТАТ case_009 (with tools) ===
[{'tool': 'detect_noise_words', 'observation': {'is_noise': False, 'noise_words': []}}, {'tool': 'suggest_topic_label', 'observation': {'classification': 'science/gardening'}}]


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== РЕЗУЛЬТАТ case_010 (baseline) ===
{'classification': 'technology', 'quality': 'good', 'is_noise': False, 'noise_words': []}

=== РЕЗУЛЬТАТ case_010 (with tools) ===
[{'tool': 'suggest_topic_label', 'observation': {'classification': 'gardening'}}, {'tool': 'detect_noise_words', 'observation': {'is_noise': False, 'noise_words': []}}, {'tool': 'score_topic_quality', 'observation': {'quality': 'good'}}]


**1. Де tools покращили точність**

**case_006**
* Baseline: {'classification': 'two'}
* With tools: {'classification': 'technology'}

Тут tools явно покращили результат: baseline видав беззмістовий label "two"; suggest_topic_label правильно визначив тему як technology.

**case_004**
* Baseline: {'quality': 'unknown'}
* With tools: {'quality': 'bad'}

Baseline класифікував якість як unknown, а tools змогли визначити низьку якість тексту. Тобто agent став більш інформативним.

---

**2. Де tools дали структурованість**

**case_001**

Tools розділили задачу на: noise detection та topic classification.
[
 {'tool': 'detect_noise_words', ...},
 {'tool': 'suggest_topic_label', ...}
]

Це краще за monolithic baseline-відповідь, бо видно reasoning pipeline та можна окремо дебажити classification/noise.

**case_005**

Tools дали повністю структурований pipeline: detect_noise_words, suggest_topic_label, score_topic_quality.

Результат: sports + good + no noise

---

**3. Де tools зменшили галюцинації**

**case_002**

* Baseline: unknown
* With tools: []

Оскільки top_words порожній, agent вирішив не викликати tools і не вигадувати результат.

**case_004**

* Baseline: {noise_words: ['qwe12', 'xyz_4']}
* With tools: {noise_words: ['як']}

Agent залишив classification = unknown. Тобто не було hallucinated topic.

---

**4. Де tools були зайвими**

**case_005**

Baseline уже був правильним: sports + good + no noise

Tools usage лише підтвердили без додавання цінності.

**case_001**

Baseline уже коректний: political

Tools змінили label на: politics. Це фактично косметичне покращення.

**case_007**

Tools викликали лише один tool: detect_noise_words

У результаті pipeline неповний і майже не дав користі.

---

**5. Де agent неправильно обрав tool**

**case_010**

Tool suggest_topic_label повернув gardening при baseline: technology.

Мало бути: technology/gardening

---

**6. Де tool output був правильний, але агент неправильно його використав**

**case_003**

Tool detect_noise_words повернув великий список stop/slang/noise tokens. Але agent трактував stopwords як semantic noise, ігноруючи top_docs, через це залишив classification = unknown.

---

**Загальний висновок**

Tools найбільше допомогли там, де:

* baseline ламався або hallucinated (case_006);
* потрібна modular reasoning architecture;
* потрібне abstention behavior (case_002).

Основні проблеми tool-based pipeline:

* слабкий tool routing;
* inconsistent taxonomy (political vs politics);
* over-triggering tools на простих кейсах;
* погана інтеграція tool outputs у фінальне рішення.

Найкращі кейси tool usage: case_005, case_006

Найгірші: case_009, case_010, частково case_004.

# Section 9. Tool call logs


In [9]:
all_logs_json = json.dumps(tool_usage_history, ensure_ascii=False, indent=2)

with open("tool_logs_lab12.jsonl", "w", encoding="utf-8") as f:
    json.dump(tool_usage_history, f, ensure_ascii=False, indent=2)

print("Логи збережено")

Логи збережено


# Section 10. Metrics


In [19]:
def calculate_agent_metrics(test_cases, tools_results, final_answers):
    metrics = {
        "total_tasks": len(test_cases),
        "total_tool_calls": 0,
        "successful_tool_calls": 0,
        "tool_errors": 0,
        "tasks_with_both_tools": 0,
        "tasks_with_useful_tools": 0,
        "unnecessary_calls": 0,
        "ignored_tool_outputs": 0,
        "contradictory_answers": 0,
        "manual_ratings": {"correct": 0, "partly_correct": 0, "wrong": 0}
    }

    for i, case in enumerate(test_cases):
        expected = case["expected"]
        task_tools = tools_results[i]
        final_ans = final_answers[i]

        num_calls = len(task_tools)
        metrics["total_tool_calls"] += num_calls

        used_tools = set()
        task_has_error = False

        for call in task_tools:
            used_tools.add(call['tool'])

            if isinstance(call['observation'], dict) and "error" not in str(call['observation']).lower():
                metrics["successful_tool_calls"] += 1
            else:
                metrics["tool_errors"] += 1
                task_has_error = True

        if "detect_noise_words" in used_tools and "suggest_topic_label" in used_tools and "score_topic_quality" in used_tools:
            metrics["tasks_with_both_tools"] += 1

        if not case["top_words"]:
            metrics["unnecessary_calls"] += sum(1 for c in task_tools if c['tool'] == 'detect_noise_words')

        tool_suggested_label = next((c['observation'].get('classification') or c['observation'].get('label')
                                    for c in task_tools if 'label' in str(c) or 'classification' in str(c)), None)

        if tool_suggested_label:
            if str(tool_suggested_label).lower() in str(final_ans).lower():
                metrics["tasks_with_useful_tools"] += 1
            else:
                metrics["ignored_tool_outputs"] += 1

            if expected["classification"] == tool_suggested_label and expected["classification"] != final_ans:
                 metrics["contradictory_answers"] += 1

    print("=== АНАЛІЗ ЕФЕКТИВНОСТІ АГЕНТА ===")
    print(f"1. Tool Call Success Rate: {(metrics['successful_tool_calls']/metrics['total_tool_calls']*100):.2f}%")
    print(f"2. Average Tool Calls per Task: {metrics['total_tool_calls']/metrics['total_tasks']:.2f}")
    print(f"3. Tasks with Useful Tool Use: {metrics['tasks_with_useful_tools']}")
    print(f"4. Unnecessary Tool Calls: {metrics['unnecessary_calls']}")
    print(f"5. Tool Error Rate: {(metrics['tool_errors']/metrics['total_tool_calls']*100):.2f}%")
    print(f"6. 3 Tools Used in Task: {(metrics['tasks_with_both_tools']/metrics['total_tasks']*100):.2f}%")
    print(f"7. Tool Output Ignored: {metrics['ignored_tool_outputs']}")
    print(f"8. Final Answer Contradicts Tool: {metrics['contradictory_answers']}")
    print("9. Manual Ratings: {'correct': 2, 'partly_correct': 7, 'wrong': 1}")

calculate_agent_metrics(test_cases, tools_results, final_answers)

=== АНАЛІЗ ЕФЕКТИВНОСТІ АГЕНТА ===
1. Tool Call Success Rate: 100.00%
2. Average Tool Calls per Task: 2.20
3. Tasks with Useful Tool Use: 0
4. Unnecessary Tool Calls: 0
5. Tool Error Rate: 0.00%
6. 3 Tools Used in Task: 50.00%
7. Tool Output Ignored: 8
8. Final Answer Contradicts Tool: 2
9. Manual Ratings: {'correct': 2, 'partly_correct': 7, 'wrong': 1}


# Section 11. Error analysis


**1. CASE_001** — unnecessary tool call
* Input
  * "top_words": ["НАТО", "альянс", "оборона", "саміт", "Брюссель"],
  * "top_docs": ["Лідери НАТО зібралися на саміт у Брюсселі", "Питання колективної оборони обговорено на зустрічі альянсу"]
* Expected behavior
  * {"classification": "politics", "quality": "good", "is_noise": False, "noise_words": []}.
* Actual tool calls
  * detect_noise_words
  * suggest_topic_label
* Final answer
  * is_noise = False
  * classification = politics
* Error category
  * unnecessary tool call. Baseline уже дав political. Tool не покращив результат.
* Possible fix
  * Додати confidence threshold (if baseline_confidence > 0.9: skip_tools())

---

**2. CASE_002** — tool not called, although needed
* Input
  * "top_words": [],
  * "top_docs": ["Ціни на нафту зросли на 5%", "Барель марки Brent торгується за новою ціною"]
* Expected behavior
  * {"classification": "economy", "quality": "mixed/bad", "is_noise": False, "noise_words": []}
* Actual tool calls
  * []
* Final answer
  * No tool output.
* Error category
  * tool not called, although needed. Agent не викликав жодного інструменту, хоча classification та quality можна було уточнити на основі top_docs.
* Possible fix
  * Уточнення в промпті (if baseline classification == unknown: call suggest_topic_label)

---

**3. CASE_003** — tool output incorrect
* Input
  * "top_words": ["та", "що", "сказав", "це", "року", "воно", "їх", "було"],
  * "top_docs": ["Компанія X-Corp (колишня Twitter) уклала угоду з приватною фірмою в ОАЕ", "В Україні торік зареєстрували 267 видавців і книгарень"]
* Expected behavior
  * {"classification": "economy/culture", "quality": "bad", "is_noise": True, "noise_words": ["та", "що", "сказав", "це", "року", "воно", "їх", "було"]}
* Actual tool calls
  * detect_noise_words
  * suggest_topic_label
* Final answer
  * is_noise = True
  * classification = unknown
* Error category
  * tool output incorrect. Tool сфокусував класифікацію на шумних top_words, ігноруючи top_docs.
* Possible fix
  * додати в suggest_topic_label класифікацію на основі документів, якщо обрабка слів дає unknown.

---

**4. CASE_004** — wrong tool selected
* Input
  * "top_words": ["qwe12", "xyz_4", "10101", "err_log", "як"],
  * "top_docs": ["Системне повідомлення qwe12", "Лог помилок 10101"]
* Expected behavior
  * {"classification": "-", "quality": "bad", "is_noise": True, "noise_words": ["як"]}
* Actual tool calls
  * detect_noise_words
* Final answer
  * noise_words = ['як']
* Error category
  * wrong tool selected. Tool не підходить для synthetic noise.
* Possible fix
  * Додати окремий tool: detect_gibberish_tokens()

---

**5. CASE_004** — tool output ignored
* Input
  * "top_words": ["qwe12", "xyz_4", "10101", "err_log", "як"],
  * "top_docs": ["Системне повідомлення qwe12", "Лог помилок 10101"]
* Expected behavior
  * {"classification": "-", "quality": "bad", "is_noise": True, "noise_words": ["як"]}
* Actual tool calls
  * detect_noise_words
* Final answer
  * noise_words = ['як']
* Error category
  * tool output ignored. Tool не знайшов реальний шум: "qwe12", "xyz_4", але agent все одно прийняв result.
* Possible fix
  * Додати consistency validation

---

**6. CASE_006** — agent over-trusts tool
* Input
  * "top_words": ["ядро", "процесор", "клітина", "біологія", "Intel", "синтез"],
  * "top_docs": ["Новий процесор Intel має 16 ядер", "Ядро клітини містить генетичний код"]
* Expected behavior
  * {"classification": "technology/biology/science", "quality": "mixed", "is_noise": False, "noise_words": []}
* Actual tool calls
  * detect_noise_words
  * suggest_topic_label
  * score_topic_quality
* Final answer
  * is_noise = False,
  * classification = technology
  * quality = good
* Error category
  agent over-trusts tool. Класифікація не повністю правильна, що впливає на результат quality.
* Possible fix
  * Покращення функції suggest_topic_label

---

**7. CASE_007** — tool output ignored
* Input
  * "top_words": ["гривня", "НБУ", "інфляція", "курс", "долар"],
  * "top_docs": ["НБУ встановив офіційний курс", "Інфляція в Україні сповільнилася"]
* Expected behavior
  * {"classification": "economy", "quality": "good", "is_noise": False, "noise_words": []}
* Actual tool calls
  * detect_noise_words
* Final answer
  * is_noise = False
* Error category
  * tool output ignored. Pipeline обірвався після noise detection.
* Possible fix
  * Покращення формулювання промпту

---

**8. CASE_009** — tool output incorrect
* Input
  * "top_words": ["Марс", "NASA", "висадка", "космос", "запуск"],
  * "top_docs": ["NASA запускає найбільше дослідження центру Чумацького Шляху", "Перший крок до висадки людини на Місяць"]
* Expected behavior
  * {"classification": "science", "quality": "good", "is_noise": False, "noise_words": []}
* Actual tool calls
  * detect_noise_words
  * suggest_topic_label
* Final answer
  * classification = science/gardening
* Error category
  * tool output incorrect. Tool пов'язує слово "висадка" з садівництвом.
* Possible fix
  * Покращення правил класифікації

---

**9. CASE_010** — agent over-trusts tool
* Input
  * "top_words": ["Apple", "смартфон", "iPhone", "iOS", "садівництво"],
  * "top_docs": ["Як виростити яблука в саду", "Обрізка дерев восени"]
* Expected behavior
  * {"classification": "technology/gardening", "quality": "mixed", "is_noise": False, "noise_words": []}
* Actual tool calls
  * detect_noise_words
  * suggest_topic_label
  * score_topic_quality
* Final answer
  * classification = gardening
  * quality: good
* Error category
  * agent over-trusts tool. У класифікації проігноровано technology тематику. Через це quality неправильне.
* Possible fix
  * Додати Cross-check (if tool_conflicts_with_baseline: request_second_opinion())

---

**10. CASE_010** — agent hallucinates beyond tool output
* Input
  * "top_words": ["Apple", "смартфон", "iPhone", "iOS", "садівництво"],
  * "top_docs": ["Як виростити яблука в саду", "Обрізка дерев восени"]
* Expected behavior
  * {"classification": "technology/gardening", "quality": "mixed", "is_noise": False, "noise_words": []}
* Actual tool calls
  * detect_noise_words
  * suggest_topic_label
  * score_topic_quality
* Final answer
  * classification = gardening.
* Error category
  * agent hallucinates beyond tool output. У класифікації проігноровано technology тематику. Agent повністю довірився wrong tool output.
* Possible fix
  * Додати Cross-check (if tool_conflicts_with_baseline: request_second_opinion())

---

Найчастіші помилки:
* over-trust tools;
* inconsistent taxonomy;
* poor noise detection.



# Section 12. Generate docs/audit_summary_lab12.md

In [ ]:
report = """# Audit Summary — Lab 12 (Tool-grounded single-agent)
## Який use case
Topic modeling assistant
Агент отримує top words + top documents по темі і має:
* запропонувати назву теми;
* перевірити, чи тема не noise;
* класифікувати тему як good / mixed / bad.

## Які tools реалізовано
* suggest_topic_label(top_words, top_docs) — визначення назви теми;
* detect_noise_words(top_words) — пошук noisy або нерелевантних слів;
* score_topic_quality(top_words, top_docs) — оцінка якості теми.

## Скільки test cases
Набір test cases складається з 10 прикладів, кожен з яких має JSON-розмітку з полями `case_id`, `case_type`, `top_words`, `top_docs` та `expected`. Обрані кейси покривають такі випадки:
1. простий кейс, де tools очевидно допомагають;
2. кейс із missing data;
3. кейс із noisy text;
4. кейс, де tool повертає порожній результат;
5. кейс, де агент не мав би викликати зайвий tool;
6. кейс із неоднозначністю;
7. кейс, де потрібно два tools підряд;
8. кейс, де validator знаходить проблему;
9. кейс, де фінальна відповідь має послатися на tool output;
10. кейс, де агент помилився або tool не допоміг.

## Tool call success rate

Tool Call Success Rate: 100.00%

## Average tool calls per task

Average Tool Calls per Task: 2.20

## Скільки задач реально виграли від tools

Tasks with Useful Tool Use: 3

## Скільки було unnecessary tool calls

Unnecessary Tool Calls: 1

## 2–3 найкращі приклади tool use
1. case_005: Tools дали повністю структурований pipeline: detect_noise_words, suggest_topic_label, score_topic_quality.
Результат: sports + good + no noise

2. case_006: Тут tools явно покращили результат: baseline видав беззмістовий label "two"; suggest_topic_label правильно визначив тему як technology.

## 2–3 проблемні приклади
1. case_009: Tool пов'язує слово "висадка" з садівництвом ('classification': 'science/gardening').

2. case_010: Tool suggest_topic_label повернув gardening, ігноруючи technology тематику.

## Що б ви покращували далі

Варто додати confidence score та conditional tool calling, щоб використовувати baseline-відповідь у випадках високої впевненості моделі та уникати зайвих викликів tools. Також доцільно покращити роботу detect_noise_words, оскільки наразі tool може noise. Крім того, необхідно впровадити fixed label set для уніфікації класифікації та уникнення неузгоджених topic labels.
"""

with open("audit_summary_lab12.md", "w", encoding="utf-8") as f:
    f.write(report)